In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, r2_score

In [3]:
# step 1: load and clean data
df = pd.read_csv('Housing.csv')

# converting text yes/no to 1 and 0 so the model can read it
cols_to_map = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']
for col in cols_to_map:
    df[col] = df[col].map({'yes': 1, 'no': 0})

df['furnishingstatus'] = df['furnishingstatus'].map({'furnished': 2, 'semi-furnished': 1, 'unfurnished': 0})
df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,1,0,0,0,1,2,1,2
1,12250000,8960,4,4,4,1,0,0,0,1,3,0,2
2,12250000,9960,3,2,2,1,0,1,0,0,2,1,1
3,12215000,7500,4,2,2,1,0,1,0,1,3,1,2
4,11410000,7420,4,1,2,1,1,1,0,1,2,0,2


In [4]:
# separate features and target
X = df.drop('price', axis=1).values
y = df['price'].values

# scaling 
def normalize(X_data):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_data)
    return X_scaled

X_scaled = normalize(X)

In [5]:
print("---- SIMPLE LINEAR REGRESSION ----")
# just using the first column (area) for simple regression
X_simple = X_scaled[:, 0].reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(X_simple, y, test_size=0.2, random_state=42)

model_simple = LinearRegression()
model_simple.fit(X_train, y_train)
preds = model_simple.predict(X_test)
print("MSE:", mean_squared_error(y_test, preds))
print("R2:", r2_score(y_test, preds))

kf = KFold(n_splits=5, shuffle=True, random_state=42)
r2_scores = []
for train_idx, test_idx in kf.split(X_simple):
    X_tr, X_te = X_simple[train_idx], X_simple[test_idx]
    y_tr, y_te = y[train_idx], y[test_idx]
    
    temp_model = LinearRegression()
    temp_model.fit(X_tr, y_tr)
    r2_scores.append(r2_score(y_te, temp_model.predict(X_te)))

print("Average K-Fold R2:", np.mean(r2_scores))

---- SIMPLE LINEAR REGRESSION ----
MSE: 3675286604768.1855
R2: 0.27287851871974633
Average K-Fold R2: 0.22624257978867984


In [6]:
print("\n---- MULTIPLE LINEAR REGRESSION ----")
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

model_multi = LinearRegression()
model_multi.fit(X_train, y_train)
preds_m = model_multi.predict(X_test)
print("MSE:", mean_squared_error(y_test, preds_m))
print("R2:", r2_score(y_test, preds_m))

r2_scores_m = []
for train_idx, test_idx in kf.split(X_scaled):
    X_tr, X_te = X_scaled[train_idx], X_scaled[test_idx]
    y_tr, y_te = y[train_idx], y[test_idx]
    
    temp_model = LinearRegression()
    temp_model.fit(X_tr, y_tr)
    r2_scores_m.append(r2_score(y_te, temp_model.predict(X_te)))

print("Average K-Fold R2:", np.mean(r2_scores_m))


---- MULTIPLE LINEAR REGRESSION ----
MSE: 1771751116594.04
R2: 0.6494754192267793
Average K-Fold R2: 0.6318615631007684
